In [ ]:
%%bash
version='2025-07-21'
wget -qN https://github.com/xinehc/sarg-curation/archive/refs/tags/$version.tar.gz -P ../tmp
tar -xf ../tmp/$version.tar.gz -C ../tmp
mv ../tmp/sarg-curation-$version/sarg_ref.fa ../tmp/sarg_old.fa
mv ../tmp/sarg-curation-$version/misc/evidence.tsv ../tmp/evidence_old.tsv

In [ ]:
import pandas as pd
old = pd.read_table('../tmp/evidence_old.tsv').set_index('evidence').subtype.to_dict()
new = pd.read_table('../misc/evidence.tsv').set_index('evidence').subtype.to_dict()
name = pd.read_table('../misc/evidence.tsv').set_index('evidence').name.to_dict()

In [ ]:
tmp = []
for i in new:
    a = old.get(i)
    b = new.get(i)
    if a != b and a is not None:
        print(f'{a} -> {b}')
    elif a is None:
        tmp.append([i, name.get(i), b])

print('\n')
for i in tmp:
    print(f'{i[1]} -> {i[2]}')

In [ ]:
from Bio import SeqIO
import pandas as pd

old = {}
with open('../tmp/sarg_old.fa') as f:
    for record in SeqIO.parse(f, 'fasta'):
        old[record.seq] = record.id
new = {}
with open('../sarg_ref.fa') as f:
    for record in SeqIO.parse(f, 'fasta'):
        new[record.seq] = record.id

In [ ]:
deleted = []
added = []
renamed = []

for i in old:
    if i not in new:
        deleted.append(old[i].split('|', 1)[-1])

for i in new:
    if i not in old:
        added.append(new[i].split('|', 1)[-1])

for i in old:
    if i in new:
        if old[i] != new[i]:
            if old[i].split('|')[-1] == new[i].split('|')[-1]:
                renamed.append((old[i].split('|', 1)[-1], new[i].split('|', 1)[-1]))

In [ ]:
with open('../CHANGELOG.md', 'w') as f:
    f.write('# Added\n')
    for i in sorted(added):
        f.write('+ ' + i + '\n')
    f.write('\n')
    
    f.write('# Deleted\n')
    for i in sorted(deleted):
        f.write('+ ' + i + '\n')
    f.write('\n')
    
    f.write('# Renamed\n')
    for i in sorted(renamed):
        f.write('+ ' + f'{i[0]} -> {i[1]}\n')
    f.write('\n')

In [ ]:
%%bash
find ../ -name "*.ipynb" -not -path "*/\.ipynb_checkpoints/*" -exec jupyter nbconvert --log-level=ERROR --clear-output --inplace {} +